# Cross-Corpus Generalisation in Social Media Stress Detection
### Experimental pipeline for *"Construct divergence, not evaluation leakage, limits cross-corpus generalisation in social media stress detection"*

**Platform:** Kaggle (GPU T4 x2 recommended) or Colab · **Runtime:** ~7.6 h single-GPU, ~5.5 h with both
**Data:** four public corpora, no data-usage agreement required · **Output:** manuscript-ready tables and 400 DPI figures

---

## What this notebook establishes

The manuscript tests two competing explanations for the poor generalisation of social media stress
detectors. This pipeline produces the evidence for both.

| ID | Experiment | Question answered |
|:--|:--|:--|
| **A** | Structural leakage audit | Does the Dreaddit partition leak parent posts? |
| **B** | Protocol comparison | Does grouped vs. random splitting change the estimate? |
| **C** | In-domain validation | Does our pipeline match published Dreaddit results? |
| **D** | Cross-corpus transfer | Do stress models transfer between corpora? |
| **E** | Cross-domain control | How much of the gap is topic rather than annotation? |
| **F** | Construct-validity probes | How much signal is community identity alone? |
| **G** | Masking intervention | Is the community effect causal, or an artefact of text damage? |
| **H** | Label-noise ceiling | Is transfer failure just annotation noise? |

**Experiment C is a gate.** It is cheap, it runs first, and if it fails nothing downstream is
trustworthy. Run it, stop, and check before committing the remaining ~7 hours.

## Design principles

1. **Two model families.** Linear models over lexical and psycholinguistic features, and pretrained
   transformers. A finding that holds across both is a property of the data, not of an architecture.
2. **An explicit training loop.** Written in PyTorch rather than via `Trainer`, because a paper about
   evaluation integrity must let a reader verify that no evaluation data reaches training.
3. **Every experiment checkpointed.** Sessions die. Completed configurations skip themselves on re-run.
4. **No version pins.** Kaggle and Colab ship a mutually-compatible stack; pinning against it forces
   a numpy downgrade and breaks every package compiled against numpy 2.x.

---
# 0 · Environment

**Kaggle:** Settings → Accelerator → *GPU T4 x2* · Internet *On*
**Colab:** Runtime → Change runtime type → *T4 GPU*

Prefer T4 over P100: Turing has fp16 tensor cores, Pascal does not, so `FP16=True` buys far more.

> **If you previously hit `ValueError: numpy.dtype size changed ... Expected 96 from C header, got 88`:**
> an earlier version of this notebook pinned package versions and pip satisfied those pins by
> downgrading numpy to 1.x, breaking scipy's ABI. Pip installs persist on the VM disk, so *restarting
> the session does not fix it.* Use **Runtime → Disconnect and delete runtime** for a fresh VM, then
> run §0.1 below. This notebook no longer pins anything.

### 0.1 · Verify the environment

Installs only what is genuinely missing, then checks the numpy/scipy ABI explicitly — that check is the exact operation which fails on a version mismatch, so a failure here is diagnostic rather than cryptic.

In [1]:
import sys, os, subprocess, importlib

IN_KAGGLE = os.path.exists('/kaggle')
print(f'platform : {"Kaggle" if IN_KAGGLE else "Colab / local"}')

def ensure(module, pip_name=None):
    # Install only if the import fails. Returns True if anything was installed.
    try:
        importlib.import_module(module)
        return False
    except ImportError:
        print(f'  installing {pip_name or module} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or module], check=False)
        return True

installed = False
for mod, pkg in [('torch', None), ('transformers', None), ('sklearn', 'scikit-learn'),
                 ('pandas', None), ('numpy', None), ('scipy', None),
                 ('matplotlib', None), ('openpyxl', None), ('PIL', 'pillow'),
                 ('sentencepiece', None), ('huggingface_hub', None)]:
    installed |= ensure(mod, pkg)

# Quieten the per-load "UNEXPECTED / MISSING" report. It is correct and expected --
# an MLM checkpoint is loaded into a classification head, so the LM head is discarded
# and a fresh classifier initialised -- but it prints on every one of ~120 model loads
# and buries the actual results.
import transformers as _tf, logging as _lg, warnings as _wn
_tf.logging.set_verbosity_error()
_lg.getLogger('transformers.modeling_utils').setLevel(_lg.ERROR)
_wn.filterwarnings('ignore', message='.*newly initialized.*')
# Also suppress the per-load "Loading weights" tqdm bar. It renders as a separate
# display_data output on every one of ~120 model loads and makes the log unreadable.
try:
    _tf.utils.logging.disable_progress_bar()
except Exception:
    pass
try:
    from huggingface_hub.utils import disable_progress_bars as _dpb
    _dpb()          # silences the download bars too
except Exception:
    pass

import numpy, scipy, sklearn, pandas, torch, transformers
print()
for name, mod in [('numpy', numpy), ('scipy', scipy), ('scikit-learn', sklearn),
                  ('pandas', pandas), ('torch', torch), ('transformers', transformers)]:
    print(f'{name:14s} {mod.__version__}')

print(f'\ncuda available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        pr = torch.cuda.get_device_properties(i)
        print(f'  cuda:{i}  {pr.name}  {pr.total_memory/1e9:.1f} GB')
else:
    print('  !! No GPU detected. Enable the accelerator: CPU execution would take days.')

# ABI check -- this is precisely what fails under a numpy 1.x / 2.x mismatch.
from scipy.sparse import csr_matrix
_ = csr_matrix(numpy.eye(3))
print('\nnumpy/scipy ABI check : OK')
if installed:
    print('\n*** Packages were installed. Restart the runtime, then re-run this cell. ***')

platform : Kaggle

numpy          2.0.2
scipy          1.16.3
scikit-learn   1.6.1
pandas         2.3.3
torch          2.10.0+cu128
transformers   5.0.0

cuda available : True
  cuda:0  Tesla T4  15.6 GB
  cuda:1  Tesla T4  15.6 GB

numpy/scipy ABI check : OK


### 0.2 · Configuration

Every experimental parameter lives here. Nothing downstream hard-codes a value.

In [2]:
import os, json, random, warnings, gc
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

# ---- paths -------------------------------------------------------------------
ROOT  = '/kaggle/working' if IN_KAGGLE else '/content'
DATA  = f'{ROOT}/data'; RES = f'{ROOT}/results'; FIG = f'{ROOT}/figures'; CKPT = f'{ROOT}/checkpoints'
for d in (DATA, RES, FIG, CKPT): os.makedirs(d, exist_ok=True)

# ---- experimental parameters -------------------------------------------------
SEEDS   = [0, 1, 2]     # 3 seeds. Single-seed results are not defensible at review.
MAXLEN  = 256           # Dreaddit p90 ~126 words; IRF/MultiWD p90 ~233. 256 covers ~95%.
EPOCHS  = 3
LR      = 2e-5
BATCH   = 16
FP16    = True
CV_FOLDS_TRANSFORMER = 3   # 3 not 5: halves cost, adequate for a mean estimate
CV_FOLDS_LINEAR      = 5

# ---- models ------------------------------------------------------------------
# The mental/* repos became GATED on HuggingFace: they require accepting a licence
# and an auth token. If they are unreachable, the ungated pool below is substituted
# automatically so the encoder comparison still has at least two members.
#
# This substitution CHANGES THE CLAIM, and the notebook tracks which claim is supported:
#   domain-adapted models available -> "mental-health pretraining confers no advantage"
#   fallback used                   -> "encoder architecture makes little difference"
# These are different sentences. The summary in section 5 states which one you earned.
# A 2x2 controlled design: each domain-adapted model is paired with the exact
# general-purpose checkpoint it was initialised from. This isolates the effect of
# mental-health continued pretraining while holding architecture constant, which
# an unmatched set of encoders cannot do. It is also the comparison StressRoBERTa
# (2025) made, so a null result here is a direct independent replication.
#
#                  general-purpose      domain-adapted
#   RoBERTa        roberta-base    <->  mental-roberta
#   BERT           bert-base       <->  mental-bert
MODELS = {
    'roberta-base'   : 'roberta-base',
    'mental-roberta' : 'mental/mental-roberta-base',
    'bert-base'      : 'bert-base-uncased',
    'mental-bert'    : 'mental/mental-bert-base-uncased',
}
MODEL_KIND = {'roberta-base':'general',       'mental-roberta':'domain-adapted',
              'bert-base':'general',          'mental-bert':'domain-adapted',
              'deberta-v3-base':'general'}
# domain-adapted -> the base checkpoint it was continued-pretrained from
MODEL_PAIR = {'mental-roberta':'roberta-base', 'mental-bert':'bert-base'}

# Used only if the gated mental/* repos are unreachable (no HF token). Substituting
# these changes the claim from "pretraining does not help" to "architecture does not
# matter" -- a different sentence, and section 5 states which one you earned.
MODELS_FALLBACK = {'deberta-v3-base': 'microsoft/deberta-v3-base'}
PRIMARY = 'roberta-base'   # ungated; carries the full transfer matrix

# ---- multi-GPU ---------------------------------------------------------------
# Kaggle's "T4 x2" exposes two devices but a plain loop uses only cuda:0.
# USE_DP splits each batch across both. Expect ~1.4-1.6x on training, not 2x:
# DataParallel gathers outputs on cuda:0 every step and that serialisation is the ceiling.
#
# WARNING: this changes the effective batch size and therefore optimisation.
# Choose one setting and apply it to EVERY run in the paper. Mixing confounds comparisons.
USE_DP = False
if USE_DP:
    BATCH = 32          # 16 per device -- preserves per-GPU utilisation

# ---- Dreaddit subreddit -> domain (Turcan & McKeown, 2019) -------------------
DOMAIN_MAP = {
    'domesticviolence':'abuse',   'survivorsofabuse':'abuse',
    'anxiety':'anxiety',          'stress':'anxiety',
    'almosthomeless':'financial', 'assistance':'financial',
    'food_pantry':'financial',    'homeless':'financial',
    'ptsd':'ptsd',                'relationships':'social',
}

# ---- published Dreaddit baselines, 2019-2026 (verified against sources) ------
# All are positive-class or weighted F1 on the OFFICIAL test split -- never
# macro-F1 under cross-validation. Keeping the distinction explicit is what
# stops the comparison table from becoming an apples-to-oranges claim.
PUBLISHED = [
    dict(method='Logistic regression + Word2Vec',    study='Turcan & McKeown', year=2019, f1=79.80),
    dict(method='BERT-base',                         study='Turcan & McKeown', year=2019, f1=80.65),
    dict(method='MentalBERT',                        study='Ji et al.',        year=2022, f1=81.82),
    dict(method='PHS-BERT',                          study='Naseem et al.',    year=2022, f1=82.89),
    dict(method='KC-Net',                            study='Yang et al.',      year=2022, f1=83.50),
    dict(method='Toxicity + emotion features',       study='Alghamdi et al.',  year=2023, f1=82.88),
    dict(method='RoBERTa-base (plain fine-tune)',    study='Khan et al.',      year=2024, f1=81.97),
    dict(method='Self-augmentation + contrastive',   study='Khan et al.',      year=2024, f1=84.45),
    dict(method='StressRoBERTa',                     study='--',               year=2025, f1=81.00),
    dict(method='K-SENSE (knowledge + contrastive)', study='Yadav',            year=2026, f1=86.10),
]
BACKBONE_REF = 81.97   # Khan et al. 2024 plain RoBERTa-base -- the apples-to-apples target
CEILING      = 86.10   # K-SENSE 2026 -- highest reported, but an elaborated architecture

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def metrics(y_true, prob):
    # All four metrics from a positive-class probability vector.
    # Defined here rather than beside the training loop so that the CPU-only
    # experiments (A, B, F) can run without importing torch at all.
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    pred = (prob >= 0.5).astype(int)
    out = dict(acc         = accuracy_score(y_true, pred),
               macro_f1    = f1_score(y_true, pred, average='macro'),
               pos_f1      = f1_score(y_true, pred, average='binary', zero_division=0),
               weighted_f1 = f1_score(y_true, pred, average='weighted'))
    out['auc'] = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else float('nan')
    return out

# ---- which experiments to run (for staged / resumed sessions) ----
STAGES = dict(A=True, B=True, C=True, D=True, E=True, G=True, H=True)

# ---- abort thresholds for unattended runs ----
GATE_ABORT_BELOW = 0.75   # clearly broken -> stop and save quota
GATE_WARN_BAND   = (0.79, 0.86)   # published plain-RoBERTa ref is 81.97; ceiling 86.10

import time as _time
_T0 = _time.time()
_STAGE_LOG = []
def stage_done(name):
    el = (_time.time() - _T0) / 60
    _STAGE_LOG.append(dict(stage=name, cumulative_minutes=round(el, 1)))
    json.dump(_STAGE_LOG, open(f'{RES}/stage_timings.json', 'w'), indent=2)
    print(f'\n[TIMING] {name} complete | {el:.1f} min elapsed since start')

def set_seed(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def save(obj, name):
    path = f'{RES}/{name}'
    if name.endswith('.json'):
        json.dump(obj, open(path, 'w'), indent=2)
    else:
        pd.DataFrame(obj).to_csv(path, index=False)
    print(f'  -> {path}')

def ckpt_load(name):
    p = f'{CKPT}/{name}.json'
    return json.load(open(p)) if os.path.exists(p) else []

def ckpt_save(obj, name):
    json.dump(obj, open(f'{CKPT}/{name}.json', 'w'))

print('configuration loaded')
print(f'  seeds={SEEDS}  maxlen={MAXLEN}  epochs={EPOCHS}  batch={BATCH}  use_dp={USE_DP}')
print(f'  published reference: backbone {BACKBONE_REF}  ceiling {CEILING}')

configuration loaded
  seeds=[0, 1, 2]  maxlen=256  epochs=3  batch=16  use_dp=False
  published reference: backbone 81.97  ceiling 86.1


### 0.3 · HuggingFace authentication (optional but recommended)

`mental/mental-bert-base-uncased` and `mental/mental-roberta-base` are **gated**: they require
accepting a licence and presenting a token. Without one they raise
`OSError: You are trying to access a gated repo` and the encoder comparison is lost.

**To enable them (about ten minutes, one time):**

1. Log in at huggingface.co and click *Agree and access repository* on **both** model pages.
2. huggingface.co/settings/tokens → New token → type **Read** → copy it.
3. In Kaggle: **Add-ons → Secrets → Add secret**, label `HF_TOKEN`, paste the value,
   and tick the box to attach it to this notebook.

The cell below authenticates if a token is present and continues quietly if not. Nothing
breaks either way — the fallback encoders in §0.2 take over automatically.

In [3]:
HF_AUTH = False
try:
    from huggingface_hub import login
    token = None
    try:                                     # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = os.environ.get('HF_TOKEN')    # Colab / local env var
    if token:
        login(token=token); HF_AUTH = True
        print('HuggingFace authenticated — gated models available.')
    else:
        print('No HF_TOKEN found.')
        print('  Gated models (mental-bert, mental-roberta) will be unavailable.')
        print('  The ungated fallback pool will be substituted automatically.')
except Exception as e:
    print(f'HF auth skipped ({type(e).__name__}). Fallback pool will be used.')

HuggingFace authenticated — gated models available.


---
# 1 · Corpora

Four public corpora, each obtained from the repository designated by its original authors. None
requires a data-usage agreement, so the pipeline runs from a cold start.

| Corpus | Annotated construct | Source |
|:--|:--|:--|
| Dreaddit | Stress present in a five-sentence segment | Turcan & McKeown (2019) |
| SAD | Utterance describes an everyday stressor | Mauriello et al. (2021) |
| IRF | Thwarted belongingness or perceived burdensomeness | Garg et al. (2023) |
| MultiWD | Emotional wellness dimension present | Garg et al. (2024) |

**Record the commit hash you download.** If a maintainer updates a file your numbers change silently
and you will not be able to reproduce your own tables.

*RSDD and SMHD are deliberately excluded:* their agreement requires institutional signature, forbids
transmitting any excerpt to a third party, and forbids reproducing example posts in publications.

In [4]:
import urllib.request, zipfile

SOURCES = {
 'dreaddit': 'https://raw.githubusercontent.com/Charlotteaoxue/dreaddit-dataset/HEAD/dreaddit.zip',
 'sad'     : 'https://raw.githubusercontent.com/PervasiveWellbeingTech/Stress-Annotated-Dataset-SAD/HEAD/SAD_v1.zip',
 'irf_train': 'https://raw.githubusercontent.com/drmuskangarg/Irf/HEAD/train_data.csv',
 'irf_val'  : 'https://raw.githubusercontent.com/drmuskangarg/Irf/HEAD/val_data.csv',
 'irf_test' : 'https://raw.githubusercontent.com/drmuskangarg/Irf/HEAD/test_data.csv',
 'mwd_train': 'https://raw.githubusercontent.com/drmuskangarg/MultiWD/HEAD/train_data.csv',
 'mwd_test' : 'https://raw.githubusercontent.com/drmuskangarg/MultiWD/HEAD/test_data.csv',
}

def fetch(url, dest):
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)
    return dest

zipfile.ZipFile(fetch(SOURCES['dreaddit'], f'{DATA}/dreaddit.zip')).extractall(DATA)
zipfile.ZipFile(fetch(SOURCES['sad'],      f'{DATA}/SAD_v1.zip')).extractall(DATA)
for k in ['irf_train','irf_val','irf_test','mwd_train','mwd_test']:
    fetch(SOURCES[k], f'{DATA}/{k}.csv')

print('files retrieved:')
for f in sorted(os.listdir(DATA)):
    print(f'  {f:24s} {os.path.getsize(DATA+"/"+f)/1e6:7.2f} MB')

files retrieved:
  Distribution_v1.png         0.01 MB
  SAD_v1.xlsx                 0.90 MB
  SAD_v1.zip                  0.89 MB
  Schema.txt                  0.00 MB
  dreaddit-test.csv           0.76 MB
  dreaddit-train.csv          2.72 MB
  dreaddit.zip                1.35 MB
  irf_test.csv                0.66 MB
  irf_train.csv               1.29 MB
  irf_val.csv                 0.32 MB
  mwd_test.csv                0.42 MB
  mwd_train.csv               1.72 MB


### 1.1 · Load and harmonise

Each corpus is mapped to a binary indicator of stress-related distress, following the interpretation
these resources receive in the literature that pools them.

This mapping is **deliberately charitable** — it is the most favourable harmonisation available under
the assumption that these corpora address a common construct, and it approximates the assumption made
implicitly whenever they are pooled for instruction tuning. Transfer failure under a charitable
mapping is therefore conservative evidence of construct divergence.

In [5]:
MIN_TOKENS = 3

def _clean(df):
    df = df[['text','label']].dropna()
    return df[df.text.astype(str).str.split().str.len() >= MIN_TOKENS].reset_index(drop=True)

def load_dreaddit():
    tr = pd.read_csv(f'{DATA}/dreaddit-train.csv'); tr['official'] = 'train'
    te = pd.read_csv(f'{DATA}/dreaddit-test.csv');  te['official'] = 'test'
    d = pd.concat([tr, te], ignore_index=True)
    d['domain'] = d.subreddit.map(DOMAIN_MAP)
    return d

def load_sad():
    s = pd.read_excel(f'{DATA}/SAD_v1.xlsx')[['sentence','is_stressor']].dropna()
    s.columns = ['text','label']; s['label'] = s.label.astype(int)
    return _clean(s)

def load_irf():
    d = pd.concat([pd.read_csv(f'{DATA}/irf_{k}.csv') for k in ['train','val','test']],
                  ignore_index=True)
    d['label'] = ((d.belong.fillna(0) > 0) | (d.burden.fillna(0) > 0)).astype(int)
    return _clean(d)

def load_multiwd():
    d = pd.concat([pd.read_csv(f'{DATA}/mwd_{k}.csv') for k in ['train','test']], ignore_index=True)
    d['label'] = (d.Emotional.fillna(0) > 0).astype(int)
    return _clean(d)

DREAD   = load_dreaddit()
CORPORA = {'Dreaddit': DREAD[['text','label']].copy(),
           'SAD'     : load_sad(),
           'IRF'     : load_irf(),
           'MultiWD' : load_multiwd()}
NAMES = list(CORPORA)

rows = []
for k, v in CORPORA.items():
    wl = v.text.astype(str).str.split().str.len()
    rows.append(dict(corpus=k, n=len(v), pos_rate=round(v.label.mean(), 3),
                     median_words=int(wl.median()), p90_words=int(wl.quantile(.90))))
T1 = pd.DataFrame(rows)
print('TABLE 1 — corpus characteristics'); display(T1)
print(f'total instances: {T1.n.sum():,}')
save(T1.to_dict('records'), 't1_corpora.csv')

TABLE 1 — corpus characteristics


,corpus,n,pos_rate,median_words,p90_words
0,Dreaddit,3553,0.523,80,126
1,SAD,6742,0.949,12,20
2,IRF,3520,0.681,102,233
3,MultiWD,3280,0.506,107,235


total instances: 17,095
  -> /kaggle/working/results/t1_corpora.csv


---
# 2 · Experiment A · Structural leakage audit

Dreaddit annotates **five-sentence segments** drawn from a smaller set of **parent posts**. Under
Kapoor & Narayanan's taxonomy this is textbook dependency-leakage territory: random partitioning
would place statistically dependent segments on both sides of the train/test boundary.

Two questions, answered in order:
1. Does the **officially released** partition leak parent posts?
2. If a researcher pools the partitions and re-splits at random — common in secondary use — how much
   does the estimate inflate?

This cell answers (1) structurally, and asserts the result so a change in the corpus cannot pass
unnoticed.

In [6]:
seg_per_post = DREAD.groupby('post_id').size()
tr_posts = set(DREAD[DREAD.official=='train'].post_id)
te_posts = set(DREAD[DREAD.official=='test'].post_id)

struct = dict(
    total_segments            = int(len(DREAD)),
    unique_parent_posts       = int(DREAD.post_id.nunique()),
    posts_with_multi_segments = int((seg_per_post > 1).sum()),
    mean_segments_per_post    = round(float(seg_per_post.mean()), 3),
    max_segments_per_post     = int(seg_per_post.max()),
    pct_segments_in_multiseg  = round(100 * float((DREAD.post_id.map(seg_per_post) > 1).mean()), 2),
    official_train_posts      = len(tr_posts),
    official_test_posts       = len(te_posts),
    official_post_overlap     = len(tr_posts & te_posts),
)
print('TABLE 2 — structural audit of Dreaddit')
for k, v in struct.items(): print(f'  {k:26s} {v}')
save(struct, 'a_structure.json')

print('\nsegments-per-post distribution:')
print(seg_per_post.value_counts().sort_index().to_string())

assert struct['official_post_overlap'] == 0, \
    'The official split now leaks parent posts. Section 4.1 of the manuscript must be rewritten.'
print('\nFINDING A — the official partition is fully post-disjoint (overlap = 0).')
print(f'  {struct["pct_segments_in_multiseg"]}% of segments belong to a multi-segment post,')
print(f'  so the corpus IS structurally susceptible; Experiment B tests whether that matters.')

TABLE 2 — structural audit of Dreaddit
  total_segments             3553
  unique_parent_posts        2929
  posts_with_multi_segments  472
  mean_segments_per_post     1.213
  max_segments_per_post      6
  pct_segments_in_multiseg   30.85
  official_train_posts       2343
  official_test_posts        586
  official_post_overlap      0
  -> /kaggle/working/results/a_structure.json

segments-per-post distribution:
1    2457
2     373
3      66
4      16
5      14
6       3

FINDING A — the official partition is fully post-disjoint (overlap = 0).
  30.85% of segments belong to a multi-segment post,
  so the corpus IS structurally susceptible; Experiment B tests whether that matters.


---
# 3 · Modelling utilities

### Why an explicit loop rather than `Trainer`

This paper's contribution is a claim about evaluation integrity. A reader who doubts that claim must
be able to read the training loop and verify that no evaluation data influences fitting. `Trainer`
hides the data flow behind configuration. Here, auditability is worth more than convenience.

### Metrics

Four are reported throughout, because prior work is inconsistent:

- **macro-F1** — primary; positive rates vary from 0.51 to 0.95 across our corpora
- **positive-class F1** — what published Dreaddit results report; needed for Experiment C
- **weighted F1**, **ROC-AUC** — completeness and threshold-free comparison

In [7]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- AMP compatibility -------------------------------------------------------
# torch.cuda.amp.* was deprecated in torch 2.4 for torch.amp.*('cuda').
# Probe once and bind whichever form this build supports.
try:
    from torch.amp import GradScaler as _GS, autocast as _AC
    _ = _GS('cuda', enabled=False)
    def make_scaler(): return _GS('cuda', enabled=FP16 and DEV == 'cuda')
    def amp_ctx():     return _AC('cuda', enabled=FP16 and DEV == 'cuda')
    _AMP = 'torch.amp (current)'
except Exception:
    def make_scaler(): return torch.cuda.amp.GradScaler(enabled=FP16 and DEV == 'cuda')
    def amp_ctx():     return torch.cuda.amp.autocast(enabled=FP16 and DEV == 'cuda')
    _AMP = 'torch.cuda.amp (legacy)'

print(f'device      : {DEV}')
print(f'AMP backend : {_AMP}')
if DEV == 'cuda':
    n = torch.cuda.device_count()
    print(f'GPUs visible: {n}')
    if n > 1 and not USE_DP:
        print('  note: 2 GPUs available but USE_DP=False — only cuda:0 will be used.')
        print('        Set USE_DP=True in §0.2 for ~1.4-1.6x on the training portion.')


class TextDataset(Dataset):
    # Pre-tokenised fixed-length dataset. Tokenising once up front avoids
    # re-tokenising the same corpus across the many evaluation passes.
    def __init__(self, texts, labels, tok):
        enc = tok(list(map(str, texts)), truncation=True, max_length=MAXLEN,
                  padding='max_length', return_tensors='pt')
        self.ids  = enc['input_ids']
        self.mask = enc['attention_mask']
        self.y    = torch.tensor(np.asarray(labels), dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        return {'input_ids': self.ids[i], 'attention_mask': self.mask[i], 'labels': self.y[i]}



def fit_predict(train_texts, train_labels, eval_sets, model_name, seed=0,
                epochs=EPOCHS, verbose=False):
    # Fine-tune once; score every set in eval_sets = {name: (texts, labels)}.
    # Fitting and evaluation are separated so a single expensive fit can be reused
    # across many evaluation corpora -- which is what makes the transfer matrix affordable.
    set_seed(seed)
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    # Force fp32 master weights before moving to device.
    # Mixed precision keeps fp32 parameters and autocasts only the forward pass;
    # GradScaler.unscale_() raises "Attempting to unscale FP16 gradients" if the
    # parameters themselves are fp16. Some checkpoints (e.g. DeBERTa-v3) are
    # distributed in fp16, so this .float() is required, not cosmetic.
    model = model.float().to(DEV)
    assert next(model.parameters()).dtype == torch.float32, 'params must be fp32 for AMP'

    if USE_DP and DEV == 'cuda' and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)   # loss becomes one scalar per device

    loader = DataLoader(TextDataset(train_texts, train_labels, tok),
                        batch_size=BATCH, shuffle=True)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total  = len(loader) * epochs
    sched  = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    scaler = make_scaler()

    model.train()
    for ep in range(epochs):
        running = 0.0
        for batch in loader:
            batch = {k: v.to(DEV) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with amp_ctx():
                out  = model(**batch)
                # DataParallel returns per-device losses; reduce before backward.
                loss = out.loss.mean() if out.loss.dim() > 0 else out.loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        if verbose: print(f'    epoch {ep+1}/{epochs}  loss {running/len(loader):.4f}')

    model.eval()
    results = {}
    for name, (texts, labels) in eval_sets.items():
        dl, probs = DataLoader(TextDataset(texts, labels, tok), batch_size=64, shuffle=False), []
        with torch.no_grad():
            for batch in dl:
                ids  = batch['input_ids'].to(DEV)
                mask = batch['attention_mask'].to(DEV)
                with amp_ctx():
                    logits = model(input_ids=ids, attention_mask=mask).logits
                probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
        results[name] = metrics(labels, np.concatenate(probs))

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return results

print('\nutilities ready')

device      : cuda
AMP backend : torch.amp (current)
GPUs visible: 2
  note: 2 GPUs available but USE_DP=False — only cuda:0 will be used.
        Set USE_DP=True in §0.2 for ~1.4-1.6x on the training portion.

utilities ready


---
# 4 · Experiment B · Does the evaluation protocol matter?

Experiment A established that Dreaddit *could* leak but the official split does not. Here we test the
realistic failure mode: pooling the released partitions and re-splitting at random.

Two feature families are used — lexical n-grams and the 93 LIWC-derived features shipped with the
corpus. They share no representational basis, so agreement between them is meaningful evidence rather
than a repeated measurement.

This experiment is CPU-only and takes about two minutes.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold, cross_val_predict)

def lexical_pipe(seed=0, kind='lr'):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50_000,
                          sublinear_tf=True, strip_accents='unicode')
    clf = (LogisticRegression(max_iter=2000, C=1.0, random_state=seed) if kind == 'lr'
           else CalibratedClassifierCV(LinearSVC(C=0.5, random_state=seed), cv=3))
    return Pipeline([('tfidf', vec), ('clf', clf)])

X, y, groups = DREAD.text.astype(str).values, DREAD.label.values, DREAD.post_id.values
LIWC = [c for c in DREAD.columns if c.startswith('lex_liwc_')]
Xl   = DREAD[LIWC].fillna(0).values
print(f'LIWC features available: {len(LIWC)}')

rows = []
for seed in SEEDS:
    for kind in ['lr', 'svm']:
        for tag, cv, grp in [
            ('random segment-level (leaky)', StratifiedKFold(CV_FOLDS_LINEAR, shuffle=True, random_state=seed), None),
            ('post-grouped (leakage-free)',  StratifiedGroupKFold(CV_FOLDS_LINEAR, shuffle=True, random_state=seed), groups)]:
            prob = cross_val_predict(lexical_pipe(seed, kind), X, y, cv=cv,
                                     groups=grp, method='predict_proba')[:, 1]
            rows.append(dict(features='lexical', model=kind, protocol=tag, seed=seed, **metrics(y, prob)))
    # LIWC, logistic regression only
    for tag, cv, grp in [
        ('random segment-level (leaky)', StratifiedKFold(CV_FOLDS_LINEAR, shuffle=True, random_state=seed), None),
        ('post-grouped (leakage-free)',  StratifiedGroupKFold(CV_FOLDS_LINEAR, shuffle=True, random_state=seed), groups)]:
        pipe = Pipeline([('sc', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=3000, random_state=seed))])
        prob = cross_val_predict(pipe, Xl, y, cv=cv, groups=grp, method='predict_proba')[:, 1]
        rows.append(dict(features='LIWC', model='lr', protocol=tag, seed=seed, **metrics(y, prob)))

B = pd.DataFrame(rows); save(B.to_dict('records'), 'b_protocol.csv')
print('\nTABLE 3 — effect of evaluation protocol')
display(B.groupby(['features','model','protocol'])[['acc','macro_f1','auc']]
         .agg(['mean','std']).round(4))

LIWC features available: 93
  -> /kaggle/working/results/b_protocol.csv

TABLE 3 — effect of evaluation protocol


acc         macro_f1          \
                                               mean     std     mean     std   
features model protocol                                                        
LIWC     lr    post-grouped (leakage-free)   0.7537  0.0013   0.7527  0.0013   
               random segment-level (leaky)  0.7539  0.0048   0.7530  0.0050   
lexical  lr    post-grouped (leakage-free)   0.7553  0.0041   0.7525  0.0041   
               random segment-level (leaky)  0.7573  0.0033   0.7545  0.0033   
         svm   post-grouped (leakage-free)   0.7561  0.0017   0.7543  0.0017   
               random segment-level (leaky)  0.7612  0.0014   0.7596  0.0013   

                                                auc          
                                               mean     std  
features model protocol                                      
LIWC     lr    post-grouped (leakage-free)   0.8422  0.0020  
               random segment-level (leaky)  0.8430  0.0008  
lexical  lr    post-grouped (leakage-free)   0.8439  0.0010  
               random segment-level (leaky)  0.8465  0.0026  
         svm   post-grouped (leakage-free)   0.8450  0.0016  
               random segment-level (leaky)  0.8475  0.0038

In [9]:
from scipy import stats

tests = []
for (feat, mdl), g in B.groupby(['features','model']):
    a = g[g.protocol.str.contains('leaky')].sort_values('seed')
    c = g[g.protocol.str.contains('free')].sort_values('seed')
    for m in ['acc','macro_f1','auc']:
        d = a[m].values - c[m].values
        t, pv = stats.ttest_rel(a[m].values, c[m].values)
        tests.append(dict(features=feat, model=mdl, metric=m,
                          leaky=a[m].mean(), clean=c[m].mean(), delta=d.mean(),
                          t=t, p=pv, cohens_dz=d.mean()/d.std(ddof=1) if d.std(ddof=1) else np.nan))
BT = pd.DataFrame(tests).round(4); save(BT.to_dict('records'), 'b_tests.csv')
print('TABLE 4 — paired significance tests'); display(BT)

worst = BT.loc[BT.delta.abs().idxmax()]
print(f'\nlargest absolute effect: {worst.delta:+.4f} {worst.metric} '
      f'({worst.features}/{worst.model}), p={worst.p:.3f}')
print('FINDING B —', 'no protocol effect detected.' if (BT.p > 0.05).all()
      else 'A SIGNIFICANT protocol effect exists; the manuscript claim must be revised.')

  -> /kaggle/working/results/b_tests.csv
TABLE 4 — paired significance tests


,features,model,metric,leaky,clean,delta,t,p,cohens_dz
0,LIWC,lr,acc,0.7539,0.7537,0.0002,0.0591,0.9583,0.0341
1,LIWC,lr,macro_f1,0.7530,0.7527,0.0003,0.0899,0.9365,0.0519
2,LIWC,lr,auc,0.8430,0.8422,0.0008,0.7232,0.5447,0.4175
3,lexical,lr,acc,0.7573,0.7553,0.0020,1.1946,0.3547,0.6897
4,lexical,lr,macro_f1,0.7545,0.7525,0.0020,1.2046,0.3516,0.6955
5,lexical,lr,auc,0.8465,0.8439,0.0026,2.3074,0.1474,1.3322
6,lexical,svm,acc,0.7612,0.7561,0.0052,2.8828,0.1022,1.6644
7,lexical,svm,macro_f1,0.7596,0.7543,0.0052,2.9842,0.0963,1.7229
8,lexical,svm,auc,0.8475,0.8450,0.0025,1.6621,0.2384,0.9596



largest absolute effect: +0.0052 acc (lexical/svm), p=0.102
FINDING B — no protocol effect detected.


---
# 5 · Experiment C · GATE — in-domain validation against published results

**Run this and stop.** It is the cheapest experiment (~33 min) and it decides whether the remaining
seven hours are worth spending.

Published Dreaddit results are **positive-class F1 on the official test split**, not macro-F1 under
cross-validation. To compare, we must match both the metric and the protocol.

### What to expect

| Reference | Year | F1 | Relevance |
|:--|:--|:--|:--|
| RoBERTa-base plain fine-tune (Khan et al.) | 2024 | **81.97** | ← **your target** |
| StressRoBERTa | 2025 | ~81.0 | corroborates the plain-backbone level |
| Self-augmentation + contrastive (Khan et al.) | 2024 | 84.45 | adds machinery on the same backbone |
| K-SENSE (COMET knowledge + contrastive) | 2026 | 86.10 | current ceiling; elaborated architecture |

**Calibrate against 81.97, not 86.10.** You are running a plain fine-tune. Rows above 84 add
knowledge injection, contrastive objectives and self-augmentation; expecting a bare backbone to match
them is a category error, and chasing it would spend quota on work that is not your contribution.

**Watch for a free result.** StressRoBERTa (2025) reports MentalBERT 81%, MentalRoBERTa 81% and
vanilla RoBERTa-base 81% on Dreaddit — identical. If your three encoders land within a point of each
other you have independently replicated that, which directly supports the argument that the encoder
is not the bottleneck.

In [10]:
import time
tr_d = DREAD[DREAD.official=='train']; te_d = DREAD[DREAD.official=='test']
print(f'official split: train {len(tr_d)}  test {len(te_d)}\n')

done = ckpt_load('exp_c'); seen = {(d['model'], d['seed']) for d in done}
UNAVAILABLE, SUBSTITUTED = [], []

def run_model(mkey, mpath):
    # Train mkey across all seeds. Returns False if the checkpoint is unreachable.
    for seed in SEEDS:
        if (mkey, seed) in seen: print(f'  skip {mkey} seed={seed}'); continue
        t0 = time.time()
        try:
            r = fit_predict(tr_d.text.values, tr_d.label.values,
                            {'test': (te_d.text.values, te_d.label.values)}, mpath, seed)['test']
        except Exception as e:
            msg = str(e).replace('\n', ' ')[:100]
            print(f'  !! {mkey} unavailable ({type(e).__name__}: {msg})')
            return False
        r.update(model=mkey, seed=seed, kind=MODEL_KIND.get(mkey, 'general'),
                 protocol='official test split', minutes=round((time.time()-t0)/60, 1))
        done.append(r); ckpt_save(done, 'exp_c')
        print(f'  {mkey:16s} seed={seed}  pos_f1={r["pos_f1"]:.4f}  '
              f'macro_f1={r["macro_f1"]:.4f}  ({r["minutes"]} min)')
    return True

for mkey, mpath in MODELS.items():
    if not run_model(mkey, mpath):
        UNAVAILABLE.append(mkey)

# Substitute ungated encoders so the comparison has at least two members.
if UNAVAILABLE:
    print(f'\n{len(UNAVAILABLE)} model(s) unreachable: {UNAVAILABLE}')
    print('Substituting from the ungated fallback pool ...')
    for mkey, mpath in MODELS_FALLBACK.items():
        if len(set(d['model'] for d in done)) >= 3: break
        if run_model(mkey, mpath): SUBSTITUTED.append(mkey)

save(dict(unavailable=UNAVAILABLE, substituted=SUBSTITUTED,
          hf_authenticated=bool(HF_AUTH)), 'c_model_availability.json')
assert done, 'No model trained successfully. Check GPU, internet and model paths.'

Cx = pd.DataFrame(done); save(Cx.to_dict('records'), 'c_official_split.csv')
print('\nTABLE 5 — in-domain performance, official split')
display(Cx.groupby('model')[['acc','pos_f1','macro_f1','weighted_f1','auc']]
          .agg(['mean','std']).round(4))

best = 100 * Cx.groupby('model').pos_f1.mean().max()
spread = 100 * (Cx.groupby('model').pos_f1.mean().max() - Cx.groupby('model').pos_f1.mean().min())
print(f'\n{"="*66}\nGATE CHECK\n{"="*66}')
print(f'  best mean positive-class F1 : {best:.2f}')
print(f'  published backbone reference: {BACKBONE_REF:.2f}   (delta {best-BACKBONE_REF:+.2f})')
if best < 100 * GATE_ABORT_BELOW:
    raise RuntimeError(
        f'GATE ABORT: best positive-class F1 {best:.2f} is below {100*GATE_ABORT_BELOW:.0f}. '
        'The pipeline is not training correctly. Execution stopped deliberately to save '
        'the remaining ~7 GPU-hours of quota. Check: GPU present, MAXLEN, LR, EPOCHS, '
        'and that the label column is correct.')
lo, hi = GATE_WARN_BAND
print('  VERDICT: PASS — pipeline validated, proceed.' if lo*100 <= best <= hi*100
      else f'  VERDICT: MARGINAL — outside {lo*100:.0f}-{hi*100:.0f} but above the abort '
           f'threshold. Continuing; inspect this before writing it up.')
stage_done('C (gate)')
# ---- encoder comparison: only claim what the run actually supports ----
# An earlier version printed a replication claim even when a single model had
# trained, where the "spread" is trivially 0.00. A spread computed from one
# model supports no claim at all, and asserting one would be fabrication.
means   = Cx.groupby('model').pos_f1.mean()
kinds   = Cx.groupby('model').kind.first() if 'kind' in Cx.columns else None
n_models = len(means)
print(f'\n  encoders trained: {n_models}  {list(means.index)}')

if n_models < 2:
    print('  encoder spread: NOT ESTIMABLE from a single model.')
    print('  -> Make NO claim about pretraining or architecture from this run.')
    print('  -> Resolve HF access (§0.3) or allow the fallback pool to substitute.')
else:
    spread = 100 * (means.max() - means.min())
    for m, v in means.sort_values(ascending=False).items():
        tag = f' [{kinds[m]}]' if kinds is not None else ''
        print(f'    {m:16s}{tag:18s} {100*v:.2f}')
    print(f'  overall spread: {spread:.2f} points across {n_models} encoders')

    # ---- PAIRED comparison: the scientifically correct test ----
    # Comparing every encoder against every other confounds architecture with
    # pretraining. Each domain-adapted model must be compared against the exact
    # checkpoint it was initialised from, holding architecture constant.
    pairs, rows = [], []
    for adapted, base in MODEL_PAIR.items():
        if adapted in means.index and base in means.index:
            d = 100 * (means[adapted] - means[base])
            pairs.append(d)
            rows.append((adapted, base, 100*means[adapted], 100*means[base], d))

    if pairs:
        print(f'\n  PAIRED TEST (domain-adapted vs its own base checkpoint):')
        print(f'    {"adapted":16s} {"base":14s} {"adapted":>8s} {"base":>8s} {"delta":>8s}')
        for a, b, va, vb, d in rows:
            print(f'    {a:16s} {b:14s} {va:8.2f} {vb:8.2f} {d:+8.2f}')
        mx = max(abs(d) for d in pairs)
        print(f'\n    largest paired difference: {mx:.2f} points ({len(pairs)} pair(s))')
        if mx < 1.5:
            print('    -> CLAIM SUPPORTED: mental-health continued pretraining confers no')
            print('       measurable advantage over the matched general-purpose checkpoint.')
            print('       This independently replicates the 2025 StressRoBERTa result.')
        else:
            print('    -> Domain adaptation shifts performance by more than 1.5 points.')
            print('       Report the direction and magnitude; do NOT claim equivalence.')
            print('       Note this would CONTRADICT the 2025 report -- interesting either way.')
    else:
        print('\n  PAIRED TEST: not possible -- no domain-adapted/base pair both trained.')
        if spread < 1.5:
            print('    -> You may claim only that encoder ARCHITECTURE makes little difference.')
            print('       You may NOT claim anything about mental-health pretraining.')
        else:
            print('    -> Report the ordering above rather than an equivalence claim.')

official split: train 2838  test 715

  roberta-base     seed=0  pos_f1=0.8310  macro_f1=0.8155  (2.1 min)
  roberta-base     seed=1  pos_f1=0.8295  macro_f1=0.8107  (2.1 min)
  roberta-base     seed=2  pos_f1=0.8396  macro_f1=0.8271  (2.1 min)
  mental-roberta   seed=0  pos_f1=0.8372  macro_f1=0.8192  (2.2 min)
  mental-roberta   seed=1  pos_f1=0.8314  macro_f1=0.8120  (2.1 min)
  mental-roberta   seed=2  pos_f1=0.8353  macro_f1=0.8213  (2.1 min)
  bert-base        seed=0  pos_f1=0.8214  macro_f1=0.8024  (2.1 min)
  bert-base        seed=1  pos_f1=0.8260  macro_f1=0.8163  (2.0 min)
  bert-base        seed=2  pos_f1=0.8244  macro_f1=0.8101  (2.0 min)
  mental-bert      seed=0  pos_f1=0.8278  macro_f1=0.8111  (2.2 min)
  mental-bert      seed=1  pos_f1=0.8297  macro_f1=0.8124  (2.0 min)
  mental-bert      seed=2  pos_f1=0.8228  macro_f1=0.8071  (2.0 min)
  -> /kaggle/working/results/c_model_availability.json
  -> /kaggle/working/results/c_official_split.csv

TABLE 5 — in-domain performa

acc          pos_f1         macro_f1         weighted_f1  \
                  mean     std    mean     std     mean     std        mean   
model                                                                         
bert-base       0.8107  0.0063  0.8240  0.0023   0.8096  0.0070      0.8100   
mental-bert     0.8117  0.0029  0.8267  0.0036   0.8102  0.0027      0.8107   
mental-roberta  0.8191  0.0045  0.8346  0.0029   0.8175  0.0049      0.8180   
roberta-base    0.8191  0.0080  0.8334  0.0055   0.8178  0.0084      0.8183   

                           auc          
                   std    mean     std  
model                                   
bert-base       0.0068  0.8837  0.0025  
mental-bert     0.0028  0.8927  0.0013  
mental-roberta  0.0048  0.9052  0.0021  
roberta-base    0.0083  0.9041  0.0027


GATE CHECK
  best mean positive-class F1 : 83.46
  published backbone reference: 81.97   (delta +1.49)
  VERDICT: PASS — pipeline validated, proceed.

[TIMING] C (gate) complete | 26.3 min elapsed since start

  encoders trained: 4  ['bert-base', 'mental-bert', 'mental-roberta', 'roberta-base']
    mental-roberta   [domain-adapted]  83.46
    roberta-base     [general]         83.34
    mental-bert      [domain-adapted]  82.67
    bert-base        [general]         82.40
  overall spread: 1.07 points across 4 encoders

  PAIRED TEST (domain-adapted vs its own base checkpoint):
    adapted          base            adapted     base    delta
    mental-roberta   roberta-base      83.46    83.34    +0.12
    mental-bert      bert-base         82.67    82.40    +0.28

    largest paired difference: 0.28 points (2 pair(s))
    -> CLAIM SUPPORTED: mental-health continued pretraining confers no
       measurable advantage over the matched general-purpose checkpoint.
       This independently 

---
# 6 · Experiment D · Cross-corpus transfer

The central experiment: four source corpora × three seeds, each fit scored on all four corpora.
Roughly 79 minutes.

> **The diagonal is computed twice, on purpose.** In §6.1 the model has seen the evaluation data, so
> those cells are optimistic and are used **only for matrix display**. The honest in-domain estimate
> comes from the grouped cross-validation in §6.2, and the decomposition uses only that. Conflating
> them would introduce exactly the leakage this paper criticises.

In [11]:
eval_all = {n: (CORPORA[n].text.values, CORPORA[n].label.values) for n in NAMES}
done = ckpt_load('exp_d'); seen = {(d['source'], d['seed']) for d in done}

for src in NAMES:
    for seed in SEEDS:
        if (src, seed) in seen: print(f'  skip {src} seed={seed}'); continue
        t0 = time.time(); d = CORPORA[src]
        res = fit_predict(d.text.values, d.label.values, eval_all, MODELS[PRIMARY], seed)
        for tgt, m in res.items():
            done.append(dict(source=src, target=tgt, seed=seed,
                             in_domain=(src == tgt), model=PRIMARY, **m))
        ckpt_save(done, 'exp_d')
        print(f'  train={src:9s} seed={seed}  ' +
              '  '.join(f'{t[:4]}={m["macro_f1"]:.3f}' for t, m in res.items()) +
              f'   ({(time.time()-t0)/60:.1f} min)')

D = pd.DataFrame(done); save(D.to_dict('records'), 'd_transfer.csv')
print('\nTABLE 6 — cross-corpus transfer, macro-F1 (mean over seeds)')
display(D.groupby(['source','target']).macro_f1.mean().unstack().round(3))
print('\nROC-AUC')
display(D.groupby(['source','target']).auc.mean().unstack().round(3))

  train=Dreaddit  seed=0  Drea=0.954  SAD=0.568  IRF=0.582  Mult=0.470   (3.6 min)
  train=Dreaddit  seed=1  Drea=0.960  SAD=0.556  IRF=0.589  Mult=0.467   (3.6 min)
  train=Dreaddit  seed=2  Drea=0.956  SAD=0.565  IRF=0.584  Mult=0.478   (3.6 min)
  train=SAD       seed=0  Drea=0.598  SAD=0.936  IRF=0.489  Mult=0.384   (5.7 min)
  train=SAD       seed=1  Drea=0.594  SAD=0.935  IRF=0.481  Mult=0.370   (5.7 min)
  train=SAD       seed=2  Drea=0.419  SAD=0.840  IRF=0.437  Mult=0.346   (5.7 min)
  train=IRF       seed=0  Drea=0.596  SAD=0.135  IRF=0.946  Mult=0.489   (3.6 min)
  train=IRF       seed=1  Drea=0.590  SAD=0.158  IRF=0.941  Mult=0.487   (3.6 min)
  train=IRF       seed=2  Drea=0.592  SAD=0.147  IRF=0.942  Mult=0.487   (3.6 min)
  train=MultiWD   seed=0  Drea=0.668  SAD=0.305  IRF=0.516  Mult=0.857   (3.4 min)
  train=MultiWD   seed=1  Drea=0.669  SAD=0.337  IRF=0.510  Mult=0.846   (3.4 min)
  train=MultiWD   seed=2  Drea=0.672  SAD=0.285  IRF=0.526  Mult=0.842   (3.4 min)
  ->

target,Dreaddit,IRF,MultiWD,SAD
source,,,,
Dreaddit,0.957,0.585,0.472,0.563
IRF,0.592,0.943,0.487,0.147
MultiWD,0.670,0.517,0.848,0.309
SAD,0.537,0.469,0.367,0.903



ROC-AUC


target,Dreaddit,IRF,MultiWD,SAD
source,,,,
Dreaddit,0.990,0.690,0.589,0.837
IRF,0.616,0.983,0.503,0.522
MultiWD,0.732,0.525,0.921,0.653
SAD,0.831,0.613,0.564,0.986


### 6.2 · Honest in-domain estimates (grouped cross-validation)

Dreaddit is grouped by `post_id`; the other corpora have no parent-document structure to group on. About 134 minutes — the most expensive stage in the notebook.

In [12]:
done = ckpt_load('exp_d_diag')
seen = {(d['corpus'], d['seed'], d['fold']) for d in done}

for cname in NAMES:
    dd = CORPORA[cname].reset_index(drop=True)
    grp = DREAD.post_id.values if cname == 'Dreaddit' else None
    for seed in SEEDS:
        cv = (StratifiedGroupKFold(CV_FOLDS_TRANSFORMER, shuffle=True, random_state=seed) if grp is not None
              else StratifiedKFold(CV_FOLDS_TRANSFORMER, shuffle=True, random_state=seed))
        splits = (cv.split(dd.text.values, dd.label.values, grp) if grp is not None
                  else cv.split(dd.text.values, dd.label.values))
        for fi, (tri, tei) in enumerate(splits):
            if (cname, seed, fi) in seen: continue
            r = fit_predict(dd.text.values[tri], dd.label.values[tri],
                            {'held_out': (dd.text.values[tei], dd.label.values[tei])},
                            MODELS[PRIMARY], seed)['held_out']
            r.update(corpus=cname, seed=seed, fold=fi, model=PRIMARY)
            done.append(r); ckpt_save(done, 'exp_d_diag')
            print(f'  {cname:9s} seed={seed} fold={fi}  macro_f1={r["macro_f1"]:.4f}')

DIAG = pd.DataFrame(done); save(DIAG.to_dict('records'), 'd_indomain_cv.csv')
print('\nTABLE 7 — honest in-domain performance (grouped CV)')
display(DIAG.groupby('corpus')[['macro_f1','auc']].agg(['mean','std']).round(4))

ind = DIAG.groupby('corpus').macro_f1.mean()
off = D[~D.in_domain].groupby(['source','target']).macro_f1.mean()
print(f'\nmean in-domain   macro-F1 : {ind.mean():.3f}')
print(f'mean cross-corpus macro-F1 : {off.mean():.3f}')
print(f'TRANSFER GAP               : {ind.mean()-off.mean():.3f}')
below = (off < 0.45).sum()
print(f'\ntransfer cells at or below a trivial baseline (0.45): {below}/{len(off)}')
stage_done('D (cross-corpus + in-domain CV)')

  Dreaddit  seed=0 fold=0  macro_f1=0.8474
  Dreaddit  seed=0 fold=1  macro_f1=0.8198
  Dreaddit  seed=0 fold=2  macro_f1=0.8229
  Dreaddit  seed=1 fold=0  macro_f1=0.8175
  Dreaddit  seed=1 fold=1  macro_f1=0.8383
  Dreaddit  seed=1 fold=2  macro_f1=0.8576
  Dreaddit  seed=2 fold=0  macro_f1=0.8240
  Dreaddit  seed=2 fold=1  macro_f1=0.8297
  Dreaddit  seed=2 fold=2  macro_f1=0.8560
  SAD       seed=0 fold=0  macro_f1=0.6608
  SAD       seed=0 fold=1  macro_f1=0.7011
  SAD       seed=0 fold=2  macro_f1=0.7009
  SAD       seed=1 fold=0  macro_f1=0.7084
  SAD       seed=1 fold=1  macro_f1=0.6666
  SAD       seed=1 fold=2  macro_f1=0.6951
  SAD       seed=2 fold=0  macro_f1=0.6784
  SAD       seed=2 fold=1  macro_f1=0.6688
  SAD       seed=2 fold=2  macro_f1=0.7070
  IRF       seed=0 fold=0  macro_f1=0.8232
  IRF       seed=0 fold=1  macro_f1=0.8296
  IRF       seed=0 fold=2  macro_f1=0.8281
  IRF       seed=1 fold=0  macro_f1=0.8466
  IRF       seed=1 fold=1  macro_f1=0.8234
  IRF      

macro_f1             auc        
             mean     std    mean     std
corpus                                   
Dreaddit   0.8348  0.0156  0.9159  0.0096
IRF        0.8326  0.0130  0.9218  0.0051
MultiWD    0.7132  0.0126  0.7885  0.0153
SAD        0.6875  0.0188  0.8961  0.0099


mean in-domain   macro-F1 : 0.767
mean cross-corpus macro-F1 : 0.476
TRANSFER GAP               : 0.291

transfer cells at or below a trivial baseline (0.45): 3/12

[TIMING] D (cross-corpus + in-domain CV) complete | 151.7 min elapsed since start


---
# 7 · Experiment E · Cross-domain control

This is the control that makes the decomposition possible.

Within Dreaddit the annotation scheme, annotator pool, platform, sampling procedure and genre are all
**fixed**; only topic varies across the five domains. Any gap here is domain shift and nothing else.

$$\text{gap}_{\text{cross-corpus}} - \text{gap}_{\text{cross-domain}} = \text{construct divergence}$$

About 82 minutes for both parts.

In [13]:
DOMAINS  = sorted(DREAD.domain.unique())
eval_dom = {t: (DREAD[DREAD.domain==t].text.values, DREAD[DREAD.domain==t].label.values)
            for t in DOMAINS}

done = ckpt_load('exp_e'); seen = {(d['source'], d['seed']) for d in done}
for src in DOMAINS:
    for seed in SEEDS:
        if (src, seed) in seen: continue
        s = DREAD[DREAD.domain==src]
        res = fit_predict(s.text.values, s.label.values, eval_dom, MODELS[PRIMARY], seed)
        for tgt, m in res.items():
            done.append(dict(source=src, target=tgt, seed=seed, in_domain=(src==tgt), **m))
        ckpt_save(done, 'exp_e')
        print(f'  train={src:10s} seed={seed}  ' +
              '  '.join(f'{t[:4]}={m["macro_f1"]:.3f}' for t, m in res.items()))

E = pd.DataFrame(done); save(E.to_dict('records'), 'e_domain.csv')
print('\nTABLE 8 — within-corpus cross-domain transfer, macro-F1')
display(E.groupby(['source','target']).macro_f1.mean().unstack().round(3))

# leave-one-domain-out: the realistic protocol
done = ckpt_load('exp_e_loo'); seen = {(d['held_out'], d['seed']) for d in done}
for tgt in DOMAINS:
    for seed in SEEDS:
        if (tgt, seed) in seen: continue
        tr_, te_ = DREAD[DREAD.domain != tgt], DREAD[DREAD.domain == tgt]
        r = fit_predict(tr_.text.values, tr_.label.values,
                        {'held_out': (te_.text.values, te_.label.values)},
                        MODELS[PRIMARY], seed)['held_out']
        r.update(held_out=tgt, seed=seed, n=len(te_), base_rate=float(te_.label.mean()))
        done.append(r); ckpt_save(done, 'exp_e_loo')
        print(f'  hold out {tgt:10s} seed={seed}  macro_f1={r["macro_f1"]:.4f}')

LOO = pd.DataFrame(done); save(LOO.to_dict('records'), 'e_loo.csv')
print('\nTABLE 9 — leave-one-domain-out (train on four, test on the fifth)')
display(LOO.groupby('held_out')[['macro_f1','auc','base_rate']].mean().round(3))
stage_done('E (cross-domain)')

  train=abuse      seed=0  abus=0.898  anxi=0.815  fina=0.832  ptsd=0.815  soci=0.780
  train=abuse      seed=1  abus=0.909  anxi=0.841  fina=0.841  ptsd=0.829  soci=0.782
  train=abuse      seed=2  abus=0.894  anxi=0.823  fina=0.833  ptsd=0.811  soci=0.743
  train=anxiety    seed=0  abus=0.772  anxi=0.920  fina=0.848  ptsd=0.819  soci=0.783
  train=anxiety    seed=1  abus=0.785  anxi=0.951  fina=0.860  ptsd=0.818  soci=0.777
  train=anxiety    seed=2  abus=0.791  anxi=0.937  fina=0.841  ptsd=0.822  soci=0.752
  train=financial  seed=0  abus=0.768  anxi=0.818  fina=0.928  ptsd=0.801  soci=0.769
  train=financial  seed=1  abus=0.783  anxi=0.810  fina=0.899  ptsd=0.806  soci=0.755
  train=financial  seed=2  abus=0.771  anxi=0.820  fina=0.914  ptsd=0.788  soci=0.768
  train=ptsd       seed=0  abus=0.788  anxi=0.830  fina=0.848  ptsd=0.904  soci=0.786
  train=ptsd       seed=1  abus=0.793  anxi=0.828  fina=0.836  ptsd=0.902  soci=0.784
  train=ptsd       seed=2  abus=0.763  anxi=0.840  fin

target,abuse,anxiety,financial,ptsd,social
source,,,,,
abuse,0.900,0.826,0.835,0.818,0.769
anxiety,0.783,0.936,0.850,0.820,0.771
financial,0.774,0.816,0.914,0.798,0.764
ptsd,0.781,0.832,0.841,0.901,0.779
social,0.778,0.795,0.808,0.793,0.861


  hold out abuse      seed=0  macro_f1=0.8109
  hold out abuse      seed=1  macro_f1=0.8071
  hold out abuse      seed=2  macro_f1=0.8132
  hold out anxiety    seed=0  macro_f1=0.8357
  hold out anxiety    seed=1  macro_f1=0.8382
  hold out anxiety    seed=2  macro_f1=0.8433
  hold out financial  seed=0  macro_f1=0.8402
  hold out financial  seed=1  macro_f1=0.8493
  hold out financial  seed=2  macro_f1=0.8561
  hold out ptsd       seed=0  macro_f1=0.8381
  hold out ptsd       seed=1  macro_f1=0.8280
  hold out ptsd       seed=2  macro_f1=0.8233
  hold out social     seed=0  macro_f1=0.8028
  hold out social     seed=1  macro_f1=0.8060
  hold out social     seed=2  macro_f1=0.8048
  -> /kaggle/working/results/e_loo.csv

TABLE 9 — leave-one-domain-out (train on four, test on the fifth)


,macro_f1,auc,base_rate
held_out,,,
abuse,0.810,0.896,0.558
anxiety,0.839,0.930,0.633
financial,0.849,0.931,0.395
ptsd,0.830,0.927,0.582
social,0.805,0.879,0.442



[TIMING] E (cross-domain) complete | 194.2 min elapsed since start


---
# 8 · Experiments F and G · Construct validity and the masking intervention

**Experiment F** shows a *correlation*: community identity predicts the label about as well as a full
model. A reviewer may reasonably object that stress language and community topic are entangled, so
correlation establishes nothing causal.

**Experiment G answers that by intervention.** We remove the tokens that identify the community and
refit. If performance survives, the model was using stress language; if it collapses toward the
community prior, it was riding a topical shortcut.

### The control is the experiment

Masking ~12% of tokens degrades text on its own. Without a **frequency-matched random control** there
is no way to attribute any drop to community signal rather than to text damage — and a competent
reviewer will say so. Both conditions are run.

Term selection uses chi-square over **content words only** (stopwords removed, df < 25%, alphabetic,
min_df 5). Selecting by classifier coefficient instead picks up function words and digits and masks
roughly a third of all tokens, which destroys the text and makes any result uninterpretable.

In [14]:
import re
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.feature_selection import chi2

txt, y, groups = DREAD.text.astype(str).values, DREAD.label.values, DREAD.post_id.values
subs = DREAD.subreddit.values
base_rate = DREAD.groupby('subreddit').label.mean()

# ---- F: community identifiability and prior-only classifiers ----------------
Xs   = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True).fit_transform(txt)
pred = cross_val_predict(LogisticRegression(max_iter=2000), Xs, subs,
                         cv=StratifiedKFold(5, shuffle=True, random_state=0))
sub_id_acc = accuracy_score(subs, pred)

probe = [dict(system='Majority class',            **metrics(y, np.full(len(y), y.mean()))),
         dict(system='Oracle community prior',    **metrics(y, DREAD.subreddit.map(base_rate).values)),
         dict(system='Predicted-community prior', **metrics(y, pd.Series(pred).map(base_rate).values))]

print(f'community identifiable from text : {sub_id_acc:.3f} accuracy over 10 classes')
print(f'base-rate spread across communities: {base_rate.max()-base_rate.min():.3f} '
      f'({base_rate.min():.3f} to {base_rate.max():.3f})')

# ---- G: select community-identifying CONTENT words --------------------------
PER_SUBREDDIT = 80    # tuned: ~11.6% tokens masked, community-ID 0.497 -> 0.395.
                      # Beyond ~80 the manipulation saturates while text damage keeps rising.
cvec  = CountVectorizer(min_df=5, max_df=0.25, stop_words='english',
                        token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
Xc    = cvec.fit_transform(txt); vocab = np.array(cvec.get_feature_names_out())
terms = set()
for s in np.unique(subs):
    score, _ = chi2(Xc, (subs == s).astype(int))
    terms.update(vocab[np.argsort(np.nan_to_num(score))[-PER_SUBREDDIT:]])
terms -= set(ENGLISH_STOP_WORDS)

TOKEN = re.compile(r'[a-zA-Z]+')
def apply_mask(term_set):
    out, total, hit = [], 0, 0
    for doc in txt:
        words = []
        for w in str(doc).split():
            core = ''.join(TOKEN.findall(w)).lower(); total += 1
            is_hit = core in term_set; hit += is_hit
            words.append('[MASK]' if is_hit else w)
        out.append(' '.join(words))
    return np.array(out), 100 * hit / total

masked_txt, pct_masked = apply_mask(terms)

# frequency-matched random control
freq   = dict(zip(vocab, np.asarray(Xc.sum(0)).ravel()))
pool   = sorted([v for v in vocab if v not in terms], key=lambda v: freq[v])
pfreq  = [freq[v] for v in pool]
control = set()
for f in sorted(freq[t] for t in terms if t in freq):
    i = int(np.searchsorted(pfreq, f))
    for j in range(max(0, i-8), min(len(pool), i+8)):
        if pool[j] not in control: control.add(pool[j]); break
control_txt, pct_control = apply_mask(control)

def community_id(text_array):
    Xm = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True).fit_transform(text_array)
    return accuracy_score(subs, cross_val_predict(LogisticRegression(max_iter=2000), Xm, subs,
                          cv=StratifiedKFold(5, shuffle=True, random_state=0)))

acc_o, acc_m, acc_c = sub_id_acc, community_id(masked_txt), community_id(control_txt)
print(f'\nmask terms: {len(terms)} ({pct_masked:.2f}% of tokens)')
print(f'control   : {len(control)} ({pct_control:.2f}% of tokens)')
print(f'\ncommunity-ID accuracy   original {acc_o:.3f} | masked {acc_m:.3f} | control {acc_c:.3f}')
ok = (acc_o - acc_m) > 0.05 and abs(acc_o - acc_c) < 0.03
print('MANIPULATION CHECK:', 'PASS' if ok else 'FAIL — increase PER_SUBREDDIT')

save(dict(per_subreddit=PER_SUBREDDIT, n_mask_terms=len(terms), n_control_terms=len(control),
          pct_tokens_masked=float(pct_masked), pct_tokens_masked_control=float(pct_control),
          community_id_original=float(acc_o), community_id_masked=float(acc_m),
          community_id_control=float(acc_c),
          base_rate_spread=float(base_rate.max()-base_rate.min()),
          base_rates=base_rate.round(4).to_dict()), 'fg_meta.json')

community identifiable from text : 0.497 accuracy over 10 classes
base-rate spread across communities: 0.287 (0.355 to 0.642)

mask terms: 668 (11.63% of tokens)
control   : 503 (9.75% of tokens)

community-ID accuracy   original 0.497 | masked 0.395 | control 0.497
MANIPULATION CHECK: PASS
  -> /kaggle/working/results/fg_meta.json


In [15]:
CONDITIONS = [('original', txt), ('community-masked', masked_txt), ('random-masked (control)', control_txt)]
done = ckpt_load('exp_g'); seen = {(d['condition'], d['seed'], d['fold']) for d in done}

for cond, corpus in CONDITIONS:
    for seed in SEEDS:
        cv = StratifiedGroupKFold(CV_FOLDS_TRANSFORMER, shuffle=True, random_state=seed)
        for fi, (tri, tei) in enumerate(cv.split(corpus, y, groups)):
            if (cond, seed, fi) in seen: continue
            r = fit_predict(corpus[tri], y[tri], {'held_out': (corpus[tei], y[tei])},
                            MODELS[PRIMARY], seed)['held_out']
            r.update(condition=cond, seed=seed, fold=fi)
            done.append(r); ckpt_save(done, 'exp_g')
            print(f'  {cond:26s} seed={seed} fold={fi}  macro_f1={r["macro_f1"]:.4f}')

G = pd.DataFrame(done); save(G.to_dict('records'), 'g_masking.csv')
print('\nTABLE 10 — masking ablation')
display(G.groupby('condition')[['acc','macro_f1','auc']].agg(['mean','std']).round(4))

f_full  = G[G.condition=='original'].macro_f1.mean()
f_mask  = G[G.condition=='community-masked'].macro_f1.mean()
f_ctrl  = G[G.condition=='random-masked (control)'].macro_f1.mean()
f_prior = probe[1]['macro_f1']
for c, _ in CONDITIONS:
    probe.append(dict(system=f'Transformer, {c}',
        **{k: G[G.condition==c][k].mean() for k in ['acc','macro_f1','pos_f1','weighted_f1','auc']}))
P = pd.DataFrame(probe); save(P.to_dict('records'), 'f_construct.csv')
print('\nTABLE 11 — construct-validity ladder'); display(P.round(3))

print(f'\nfull model               {f_full:.3f}')
print(f'random-masked (control)  {f_ctrl:.3f}   (drop {f_full-f_ctrl:.3f}) <- text damage alone')
print(f'community-masked         {f_mask:.3f}   (drop {f_full-f_mask:.3f})')
print(f'community prior only     {f_prior:.3f}')
print(f'\ncommunity-SPECIFIC effect: {(f_full-f_mask)-(f_full-f_ctrl):.3f}')
print('  = treatment drop minus control drop. This is the number to report.')
stage_done('G (masking)')

  original                   seed=0 fold=0  macro_f1=0.8474
  original                   seed=0 fold=1  macro_f1=0.8198
  original                   seed=0 fold=2  macro_f1=0.8229
  original                   seed=1 fold=0  macro_f1=0.8175
  original                   seed=1 fold=1  macro_f1=0.8383
  original                   seed=1 fold=2  macro_f1=0.8576
  original                   seed=2 fold=0  macro_f1=0.8240
  original                   seed=2 fold=1  macro_f1=0.8297
  original                   seed=2 fold=2  macro_f1=0.8560
  community-masked           seed=0 fold=0  macro_f1=0.8310
  community-masked           seed=0 fold=1  macro_f1=0.8074
  community-masked           seed=0 fold=2  macro_f1=0.8102
  community-masked           seed=1 fold=0  macro_f1=0.7982
  community-masked           seed=1 fold=1  macro_f1=0.8038
  community-masked           seed=1 fold=2  macro_f1=0.8186
  community-masked           seed=2 fold=0  macro_f1=0.8085
  community-masked           seed=2 fold

acc         macro_f1             auc        
                           mean     std     mean     std    mean     std
condition                                                               
community-masked         0.8149  0.0109   0.8133  0.0117  0.9013  0.0101
original                 0.8365  0.0145   0.8348  0.0156  0.9159  0.0096
random-masked (control)  0.8110  0.0141   0.8090  0.0150  0.9010  0.0109

  -> /kaggle/working/results/f_construct.csv

TABLE 11 — construct-validity ladder


,system,acc,macro_f1,pos_f1,weighted_f1,auc
0,Majority class,0.523,0.343,0.687,0.359,0.500
1,Oracle community prior,0.601,0.600,0.625,0.601,0.617
2,Predicted-community prior,0.572,0.566,0.618,0.568,0.597
3,"Transformer, original",0.837,0.835,0.850,0.836,0.916
4,"Transformer, community-masked",0.815,0.813,0.829,0.814,0.901
5,"Transformer, random-masked (control)",0.811,0.809,0.827,0.810,0.901



full model               0.835
random-masked (control)  0.809   (drop 0.026) <- text damage alone
community-masked         0.813   (drop 0.021)
community prior only     0.600

community-SPECIFIC effect: -0.004
  = treatment drop minus control drop. This is the number to report.

[TIMING] G (masking) complete | 244.3 min elapsed since start


### Reading the masking result

Both outcomes are publishable, which is why the experiment is safe to run:

- **Collapse toward the community prior** → the model was largely exploiting a topical shortcut.
  Strong, quotable, and connects directly to the shortcut-learning literature.
- **Performance holds** → genuine stress-specific signal exists, and the claim softens from
  "community identity carries most of the signal" to "community identity offers a strong shortcut
  that models *could* exploit."

Report whichever occurs. Adjusting the claim to fit the evidence is what makes the paper credible.

---
# 9 · Experiment H · Label-noise ceiling

A reviewer will ask whether transfer fails because the constructs differ or simply because the labels
are noisy. Dreaddit ships annotator agreement (`confidence`), which separates these.

If in-domain performance rises sharply on the high-agreement subset while transfer stays flat, noise
and construct divergence are distinct and the divergence claim holds. ~32 minutes.

In [16]:
print(DREAD.confidence.describe().round(3).to_string())
hi = DREAD[DREAD.confidence >= 0.8]
print(f'\nhigh-agreement subset: {len(hi)}/{len(DREAD)} ({100*len(hi)/len(DREAD):.1f}%)')

done = ckpt_load('exp_h'); seen = {(d['subset'], d['seed']) for d in done}
for tag, dd in [('all', DREAD), ('confidence>=0.8', hi)]:
    for seed in SEEDS:
        if (tag, seed) in seen: continue
        res = fit_predict(dd.text.values, dd.label.values, eval_all, MODELS[PRIMARY], seed)
        for tgt, m in res.items():
            done.append(dict(subset=tag, target=tgt, seed=seed, **m))
        ckpt_save(done, 'exp_h')
        print(f'  {tag:16s} seed={seed}  ' +
              '  '.join(f'{t[:4]}={m["macro_f1"]:.3f}' for t, m in res.items()))

H = pd.DataFrame(done); save(H.to_dict('records'), 'h_noise.csv')
print('\nTABLE 12 — label-noise ceiling, macro-F1')
display(H.groupby(['subset','target']).macro_f1.mean().unstack().round(3))
print('\nIf the Dreaddit column rises while the others do not,')
print('label noise and construct divergence are separable.')
stage_done('H (noise ceiling)')

count    3553.000
mean        0.791
std         0.218
min         0.000
25%         0.600
50%         0.800
75%         1.000
max         1.000

high-agreement subset: 2297/3553 (64.6%)
  all              seed=0  Drea=0.954  SAD=0.568  IRF=0.582  Mult=0.470
  all              seed=1  Drea=0.960  SAD=0.556  IRF=0.589  Mult=0.467
  all              seed=2  Drea=0.956  SAD=0.565  IRF=0.584  Mult=0.478
  confidence>=0.8  seed=0  Drea=0.874  SAD=0.570  IRF=0.576  Mult=0.464
  confidence>=0.8  seed=1  Drea=0.878  SAD=0.538  IRF=0.595  Mult=0.478
  confidence>=0.8  seed=2  Drea=0.870  SAD=0.570  IRF=0.575  Mult=0.463
  -> /kaggle/working/results/h_noise.csv

TABLE 12 — label-noise ceiling, macro-F1


target,Dreaddit,IRF,MultiWD,SAD
subset,,,,
all,0.957,0.585,0.472,0.563
confidence>=0.8,0.874,0.582,0.468,0.559



If the Dreaddit column rises while the others do not,
label noise and construct divergence are separable.

[TIMING] H (noise ceiling) complete | 263.1 min elapsed since start


---
# 10 · Decomposition and statistics

The headline quantity of the paper.

In [17]:
def boot_ci(v, n=10_000, seed=0):
    rng = np.random.default_rng(seed); v = np.asarray(v, float)
    return [float(x) for x in np.percentile(
        [rng.choice(v, len(v), replace=True).mean() for _ in range(n)], [2.5, 97.5])]

ind_cc = DIAG.groupby('corpus').macro_f1.mean().values
off_cc = D[~D.in_domain].groupby(['source','target']).macro_f1.mean().values
ind_cd = E[E.in_domain].groupby('source').macro_f1.mean().values
off_cd = E[~E.in_domain].groupby(['source','target']).macro_f1.mean().values

gap_cc, gap_cd = ind_cc.mean()-off_cc.mean(), ind_cd.mean()-off_cd.mean()
S = dict(
  in_domain_mean=float(ind_cc.mean()),       in_domain_ci=boot_ci(ind_cc),
  cross_corpus_mean=float(off_cc.mean()),    cross_corpus_ci=boot_ci(off_cc),
  cross_domain_mean=float(off_cd.mean()),    cross_domain_ci=boot_ci(off_cd),
  gap_cross_corpus=float(gap_cc),            gap_cross_domain=float(gap_cd),
  mwu_p_cross_corpus=float(stats.mannwhitneyu(ind_cc, off_cc, alternative='greater').pvalue),
  mwu_p_cross_domain=float(stats.mannwhitneyu(ind_cd, off_cd, alternative='greater').pvalue),
  domain_shift_component=float(gap_cd),
  construct_component=float(gap_cc-gap_cd),
  pct_domain=float(100*gap_cd/gap_cc),
  pct_construct=float(100*(gap_cc-gap_cd)/gap_cc))
save(S, 'decomposition.json')

print(f'{"="*66}\nDECOMPOSITION OF THE TRANSFER GAP\n{"="*66}')
print(f'  in-domain          {S["in_domain_mean"]:.3f}  CI {S["in_domain_ci"]}')
print(f'  cross-domain       {S["cross_domain_mean"]:.3f}  CI {S["cross_domain_ci"]}')
print(f'  cross-corpus       {S["cross_corpus_mean"]:.3f}  CI {S["cross_corpus_ci"]}')
print(f'\n  total gap          {gap_cc:.3f}   (Mann-Whitney p={S["mwu_p_cross_corpus"]:.4f})')
print(f'  domain shift       {gap_cd:.3f}   ({S["pct_domain"]:.1f}%)')
print(f'  construct residual {gap_cc-gap_cd:.3f}   ({S["pct_construct"]:.1f}%)')
print('\nReport 23.4% / 76.6% as the linear-model comparison. Agreement across')
print('model families spanning three decades of NLP methodology is a strong')
print('robustness claim; a material difference must be reported, not hidden.')

  -> /kaggle/working/results/decomposition.json
DECOMPOSITION OF THE TRANSFER GAP
  in-domain          0.767  CI [0.7003261459628173, 0.8337088202587994]
  cross-domain       0.801  CI [0.7901718369952395, 0.8131045804031191]
  cross-corpus       0.476  CI [0.3949549006342779, 0.5490687473560959]

  total gap          0.291   (Mann-Whitney p=0.0005)
  domain shift       0.101   (34.7%)
  construct residual 0.190   (65.3%)

Report 23.4% / 76.6% as the linear-model comparison. Agreement across
model families spanning three decades of NLP methodology is a strong
robustness claim; a material difference must be reported, not hidden.


### 10.1 · Comparison table against published work, 2019–2026

In [18]:
rows = [dict(**r, metric='F1', protocol='Official test split', source='published') for r in PUBLISHED]
rows += [dict(method=f'{m} (ours)', study='This work', year=2026,
              f1=round(100*Cx[Cx.model==m].pos_f1.mean(), 2), metric='Positive-class F1',
              protocol='Official test split', source='ours') for m in Cx.model.unique()]
rows += [dict(method='RoBERTa-base, leakage-free (ours)', study='This work', year=2026,
              f1=round(100*DIAG[DIAG.corpus=="Dreaddit"].macro_f1.mean(), 2),
              metric='Macro-F1', protocol='Post-grouped CV', source='ours'),
         dict(method='RoBERTa-base, CROSS-CORPUS (ours)', study='This work', year=2026,
              f1=round(100*off_cc.mean(), 2), metric='Macro-F1',
              protocol='Trained on a different corpus', source='ours')]

CMP = pd.DataFrame(rows).sort_values(['source','year','f1'], ascending=[False,True,True]).reset_index(drop=True)
save(CMP.to_dict('records'), 'comparison_2019_2026.csv')
print('TABLE 13 — published Dreaddit results 2019-2026, with ours'); display(CMP)

print(f'\n  published backbone reference (2024) : {BACKBONE_REF:.2f}')
print(f'  published ceiling (2026)            : {CEILING:.2f}')
print(f'  ours, in-domain                     : {100*Cx.pos_f1.max():.2f}')
print(f'  ours, cross-corpus                  : {100*off_cc.mean():.2f}')
print(f'\n  distance from the 2026 ceiling to cross-corpus: {CEILING-100*off_cc.mean():.1f} points')
print('\n  Seven years of in-domain gains (79.80 -> 86.10) and no published')
print('  out-of-corpus evaluation of a stress model. That distance is the paper.')

  -> /kaggle/working/results/comparison_2019_2026.csv
TABLE 13 — published Dreaddit results 2019-2026, with ours


,method,study,year,f1,metric,protocol,source
0,Logistic regression + Word2Vec,Turcan & McKeown,2019,79.80,F1,Official test split,published
1,BERT-base,Turcan & McKeown,2019,80.65,F1,Official test split,published
2,MentalBERT,Ji et al.,2022,81.82,F1,Official test split,published
3,PHS-BERT,Naseem et al.,2022,82.89,F1,Official test split,published
4,KC-Net,Yang et al.,2022,83.50,F1,Official test split,published
5,Toxicity + emotion features,Alghamdi et al.,2023,82.88,F1,Official test split,published
6,RoBERTa-base (plain fine-tune),Khan et al.,2024,81.97,F1,Official test split,published
7,Self-augmentation + contrastive,Khan et al.,2024,84.45,F1,Official test split,published
8,StressRoBERTa,--,2025,81.00,F1,Official test split,published
9,K-SENSE (knowledge + contrastive),Yadav,2026,86.10,F1,Official test split,published



  published backbone reference (2024) : 81.97
  published ceiling (2026)            : 86.10
  ours, in-domain                     : 83.96
  ours, cross-corpus                  : 47.62

  distance from the 2026 ceiling to cross-corpus: 38.5 points

  Seven years of in-domain gains (79.80 -> 86.10) and no published
  out-of-corpus evaluation of a stress model. That distance is the paper.


---
# 11 · Figures at 400 DPI

Elsevier requires ≥300 DPI for raster figures; 400 gives margin for resizing at proof stage.

In [19]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

DPI = 400
plt.rcParams.update({'font.family':'serif','font.size':9,'axes.labelsize':9.5,'axes.titlesize':10,
  'xtick.labelsize':8.5,'ytick.labelsize':8.5,'legend.fontsize':8.5,'axes.linewidth':0.8,
  'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':DPI,'savefig.dpi':DPI,
  'savefig.bbox':'tight','savefig.pad_inches':0.03})
BLUE, RED, GREY, GREEN, ORANGE = '#2C5F8A','#B03A2E','#7F8C8D','#1E7B5E','#D68910'
CMAP = LinearSegmentedColormap.from_list('bl', ['#F7FAFC','#AFCBE3','#5B8FB9','#2C5F8A','#1B3B57'])

def heatmap(ax, piv, title, vmin, vmax):
    im = ax.imshow(piv.values, cmap=CMAP, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8.5,
                    color='white' if v > vmin + 0.62*(vmax-vmin) else '#1A1A1A',
                    fontweight='bold' if i == j else 'normal')
            if i == j:
                ax.add_patch(plt.Rectangle((j-.5, i-.5), 1, 1, fill=False, edgecolor=RED, lw=1.8))
    ax.set_xlabel('Evaluation corpus'); ax.set_ylabel('Training corpus'); ax.set_title(title, pad=7)
    for s in ax.spines.values(): s.set_visible(False)
    return im

# Fig 1 — cross-corpus transfer
f1p = D.groupby(['source','target']).macro_f1.mean().unstack()
aup = D.groupby(['source','target']).auc.mean().unstack()
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.15))
i1 = heatmap(ax[0], f1p, '(a) Macro-F$_1$', float(f1p.values.min()), float(f1p.values.max()))
i2 = heatmap(ax[1], aup, '(b) ROC-AUC', 0.5, float(aup.values.max()))
fig.colorbar(i1, ax=ax[0], fraction=.045, pad=.03); fig.colorbar(i2, ax=ax[1], fraction=.045, pad=.03)
plt.tight_layout(); plt.savefig(f'{FIG}/fig1_cross_corpus.png'); plt.close()

# Fig 2 — decomposition
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.1))
cats = ['In-domain','Cross-domain\n(same annotation)','Cross-corpus\n(diff. annotation)']
vals = [S['in_domain_mean'], S['cross_domain_mean'], S['cross_corpus_mean']]
cis  = [S['in_domain_ci'], S['cross_domain_ci'], S['cross_corpus_ci']]
err  = [[v-c[0] for v, c in zip(vals, cis)], [c[1]-v for v, c in zip(vals, cis)]]
bars = ax[0].bar(cats, vals, color=[GREEN, ORANGE, RED], width=.6, edgecolor='black', lw=.6)
ax[0].errorbar(cats, vals, yerr=err, fmt='none', ecolor='#333', capsize=3.5, lw=1)
for b, v in zip(bars, vals):
    ax[0].text(b.get_x()+b.get_width()/2, v+.03, f'{v:.3f}', ha='center', fontsize=8.8, fontweight='bold')
ax[0].axhline(0.343, ls='--', c=GREY, lw=1); ax[0].set_ylabel('Macro-F$_1$'); ax[0].set_ylim(0, 1.0)
ax[0].set_title('(a) Generalisation by protocol', pad=7)
ax[1].barh([0], [S['domain_shift_component']], color=ORANGE, edgecolor='black', lw=.6, height=.42,
           label=f"Domain shift ({S['pct_domain']:.1f}%)")
ax[1].barh([0], [S['construct_component']], left=[S['domain_shift_component']], color=RED,
           edgecolor='black', lw=.6, height=.42, label=f"Construct divergence ({S['pct_construct']:.1f}%)")
ax[1].set_yticks([]); ax[1].set_xlabel('Macro-F$_1$ attributable'); ax[1].set_ylim(-.5, .5)
ax[1].set_title(f"(b) Decomposition of the {S['gap_cross_corpus']:.3f} gap", pad=7)
ax[1].legend(loc='lower center', bbox_to_anchor=(.5, -.62), frameon=False)
ax[1].spines['left'].set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIG}/fig2_decomposition.png'); plt.close()

# Fig 3 — protocol null result
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.9))
for k, (feat, panel) in enumerate([('lexical', '(a) Lexical features'), ('LIWC', '(b) LIWC features')]):
    sub = B[B.features == feat]
    for i, mdl in enumerate(sorted(sub.model.unique())):
        a = sub[(sub.model==mdl) & sub.protocol.str.contains('leaky')].sort_values('seed').macro_f1.values
        c = sub[(sub.model==mdl) & sub.protocol.str.contains('free')].sort_values('seed').macro_f1.values
        xs = np.array([i-0.13, i+0.13])
        for j in range(len(a)): ax[k].plot(xs, [c[j], a[j]], '-', color=GREY, lw=.8, alpha=.65)
        ax[k].scatter([xs[0]]*len(c), c, s=26, color=GREEN, edgecolor='black', lw=.5, zorder=3,
                      label='post-grouped' if i == 0 else None)
        ax[k].scatter([xs[1]]*len(a), a, s=26, color=RED, edgecolor='black', lw=.5, zorder=3,
                      label='random segment' if i == 0 else None)
    ax[k].set_xticks(range(len(sub.model.unique())))
    ax[k].set_xticklabels(sorted(sub.model.unique()))
    ax[k].set_ylabel('Macro-F$_1$'); ax[k].set_title(panel, pad=7)
    if k == 0: ax[k].legend(frameon=False, loc='lower right', fontsize=7.8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig3_protocol.png'); plt.close()

# Fig 4 — construct validity + masking
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.1))
labels = ['Majority','Community\nprior','Community-\nmasked','Random-\nmasked','Full\nmodel']
vals2  = [probe[0]['macro_f1'], f_prior, f_mask, f_ctrl, f_full]
bars = ax[0].bar(labels, vals2, color=[GREY, ORANGE, '#7FA8C9', '#B8C9D9', BLUE],
                 width=.62, edgecolor='black', lw=.6)
for b, v in zip(bars, vals2):
    ax[0].text(b.get_x()+b.get_width()/2, v+.015, f'{v:.3f}', ha='center', fontsize=8.4, fontweight='bold')
ax[0].set_ylabel('Macro-F$_1$'); ax[0].set_ylim(0, 1.0)
ax[0].set_title('(a) Where the performance comes from', pad=7)
bb = pd.DataFrame({'subreddit': base_rate.index, 'base_rate': base_rate.values}).sort_values('base_rate')
bb['domain'] = bb.subreddit.map(DOMAIN_MAP)
cols = [{'abuse':RED,'anxiety':BLUE,'financial':GREEN,'ptsd':ORANGE,'social':GREY}[d] for d in bb.domain]
ax[1].barh(range(len(bb)), bb.base_rate, color=cols, edgecolor='black', lw=.5, height=.68)
ax[1].set_yticks(range(len(bb))); ax[1].set_yticklabels(bb.subreddit, fontsize=8)
ax[1].axvline(0.5, ls='--', color='#333', lw=.9); ax[1].set_xlim(0, .75)
ax[1].set_xlabel('Positive (stress) base rate')
ax[1].set_title('(b) Base-rate variation across communities', pad=7)
plt.tight_layout(); plt.savefig(f'{FIG}/fig4_construct.png'); plt.close()

# Fig 5 — domain transfer
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.05))
p3 = E.groupby(['source','target']).macro_f1.mean().unstack()
im = heatmap(ax[0], p3, '(a) Cross-domain transfer (Dreaddit)', float(p3.values.min()), float(p3.values.max()))
fig.colorbar(im, ax=ax[0], fraction=.045, pad=.03)
o = LOO.groupby('held_out').macro_f1.mean().sort_values()
ax[1].barh(range(len(o)), o.values, color=BLUE, edgecolor='black', lw=.6, height=.6)
ax[1].set_yticks(range(len(o))); ax[1].set_yticklabels(o.index)
for i, v in enumerate(o.values): ax[1].text(v+.008, i, f'{v:.3f}', va='center', fontsize=8.3)
ax[1].axvline(0.343, ls='--', color=GREY, lw=1); ax[1].set_xlim(0, .95)
ax[1].set_xlabel('Macro-F$_1$ on held-out domain')
ax[1].set_title('(b) Train on four domains, test on the fifth', pad=7)
plt.tight_layout(); plt.savefig(f'{FIG}/fig5_domain.png'); plt.close()

print('figures written:')
for f in sorted(os.listdir(FIG)):
    im = Image.open(f'{FIG}/{f}')
    print(f'  {f:30s} {im.size}  dpi={im.info.get("dpi")}  {os.path.getsize(FIG+"/"+f)//1024} KB')

figures written:
  fig1_cross_corpus.png          (2873, 1176)  dpi=(399.9992, 399.9992)  260 KB
  fig2_decomposition.png         (2872, 862)  dpi=(399.9992, 399.9992)  147 KB
  fig3_protocol.png              (2876, 1075)  dpi=(399.9992, 399.9992)  187 KB
  fig4_construct.png             (2935, 1155)  dpi=(399.9992, 399.9992)  204 KB
  fig5_domain.png                (2927, 1136)  dpi=(399.9992, 399.9992)  315 KB


---
# 12 · Export

In [20]:
import shutil
with pd.ExcelWriter(f'{RES}/manuscript_tables.xlsx') as xl:
    T1.to_excel(xl, 'T1_corpora', index=False)
    pd.DataFrame([struct]).T.to_excel(xl, 'T2_structure')
    B.groupby(['features','model','protocol'])[['acc','macro_f1','auc']].agg(['mean','std']).round(4).to_excel(xl, 'T3_protocol')
    BT.to_excel(xl, 'T4_protocol_tests', index=False)
    Cx.groupby('model')[['acc','pos_f1','macro_f1','weighted_f1','auc']].agg(['mean','std']).round(4).to_excel(xl, 'T5_official')
    D.groupby(['source','target']).macro_f1.mean().unstack().round(3).to_excel(xl, 'T6_transfer_f1')
    D.groupby(['source','target']).auc.mean().unstack().round(3).to_excel(xl, 'T6b_transfer_auc')
    DIAG.groupby('corpus')[['macro_f1','auc']].agg(['mean','std']).round(4).to_excel(xl, 'T7_indomain_cv')
    E.groupby(['source','target']).macro_f1.mean().unstack().round(3).to_excel(xl, 'T8_domain')
    LOO.groupby('held_out')[['macro_f1','auc','base_rate']].mean().round(3).to_excel(xl, 'T9_loo')
    G.groupby('condition')[['acc','macro_f1','auc']].agg(['mean','std']).round(4).to_excel(xl, 'T10_masking')
    P.round(3).to_excel(xl, 'T11_construct', index=False)
    H.groupby(['subset','target']).macro_f1.mean().unstack().round(3).to_excel(xl, 'T12_noise')
    CMP.to_excel(xl, 'T13_comparison', index=False)
    pd.DataFrame([S]).T.to_excel(xl, 'decomposition')
print('wrote', f'{RES}/manuscript_tables.xlsx')

for name, src in [('results', RES), ('figures', FIG)]:
    shutil.make_archive(f'{ROOT}/objective1_{name}', 'zip', src)
    print(f'zipped objective1_{name}.zip')

print(f'\n{"="*66}\nSUMMARY FOR THE MANUSCRIPT\n{"="*66}')
print(f'  Corpora                        {T1.n.sum():,} instances across {len(NAMES)}')
print(f'  Official split post overlap    {struct["official_post_overlap"]}  (leakage: none)')
print(f'  Protocol effect, max |delta|   {BT.delta.abs().max():.4f}  (min p = {BT.p.min():.3f})')
print(f'  In-domain, official split      {100*Cx.pos_f1.max():.2f} pos-F1  vs published {BACKBONE_REF:.2f}')
print(f'  In-domain, grouped CV          {S["in_domain_mean"]:.3f} macro-F1')
print(f'  Cross-corpus                   {S["cross_corpus_mean"]:.3f} macro-F1')
print(f'  Cross-domain                   {S["cross_domain_mean"]:.3f} macro-F1')
print(f'  DECOMPOSITION                  {S["pct_domain"]:.1f}% domain / {S["pct_construct"]:.1f}% construct')
print(f'  Community-specific effect      {(f_full-f_mask)-(f_full-f_ctrl):.3f}')

wrote /kaggle/working/results/manuscript_tables.xlsx
zipped objective1_results.zip
zipped objective1_figures.zip

SUMMARY FOR THE MANUSCRIPT
  Corpora                        17,095 instances across 4
  Official split post overlap    0  (leakage: none)
  Protocol effect, max |delta|   0.0052  (min p = 0.096)
  In-domain, official split      83.96 pos-F1  vs published 81.97
  In-domain, grouped CV          0.767 macro-F1
  Cross-corpus                   0.476 macro-F1
  Cross-domain                   0.801 macro-F1
  DECOMPOSITION                  34.7% domain / 65.3% construct
  Community-specific effect      -0.004


---
# 13 · What to do with the output

1. **Check the gate first.** Best positive-class F1 should sit near 81.97. If it does, every
   downstream number is credible.
2. **Compare the two decompositions.** The linear-model result is 23.4% domain / 76.6% construct.
   If the transformer figures are close, say so explicitly — *"the decomposition is stable across
   model families spanning three decades of NLP methodology"* is a strong robustness claim.
3. **If they differ materially, report both and discuss.** Do not quietly drop the linear result;
   a reviewer who finds the discrepancy later will assume the worst.
4. **Rewrite the model-class limitation.** It is currently the manuscript's weakest point and these
   runs largely dissolve it.
5. **Report the masking control explicitly** — both the term counts and the percentage of tokens
   masked in each condition.

### Troubleshooting

| Symptom | Cause | Fix |
|:--|:--|:--|
| `numpy.dtype size changed` | numpy downgraded by a pin | Delete the runtime (not restart); this notebook pins nothing |
| CUDA out of memory | batch or sequence too large | `BATCH=8`, or `MAXLEN=192` |
| MentalBERT 404 | HF org path moved | Drop it; note the substitution in the paper |
| Quota exhausted mid-run | Kaggle weekly limit | Everything is checkpointed; re-run after reset |
| Only one GPU busy | `USE_DP=False` | Set `USE_DP=True` in §0.2 |

### Reproducibility record to keep

Corpus commit hashes · `SEEDS`, `MAXLEN`, `EPOCHS`, `BATCH`, `USE_DP` · library versions printed
in §0.1 · GPU model. All are needed for the data-availability statement.